
### MCS-382 Machine Learning Final Project

### Import the data and load packages

In [ ]:
# Yes, your multimodal model makes sense and has a clear, logical flow. Breaking the task into modular components helps to address the different aspects of the problem effectively. Here’s how each step integrates well:
#
#     Feature Extraction with Spectrograms (Model 1): Using a CNN to compress and reconstruct spectrograms is a strong approach to extract the most relevant features, reducing noise and dimensionality while retaining key information.
#
#     Aspect List Classification (Model 2): Predicting aspect lists from spectrograms bridges the gap between raw audio features and human-readable semantic labels, which is a crucial intermediary step.
#
#     Paragraph Generation with BERT (Model 3): Leveraging BERT to translate aspect lists into paragraph-style descriptions is an efficient way to model relationships between semantic labels and textual summaries.
#
#     Temporal Alignment (Model 4): Learning correspondences between spectrogram elements and aspect lists ensures that temporal dynamics are captured, enriching the generated text’s contextual accuracy.
#
#     Final Integration (Model 5): Combining the outputs from Models 3 and 4 ensures the generated description is both semantically accurate and temporally aligned with the input audio.
#
# The modular design ensures flexibility and interpretability while keeping the computational complexity manageable. Additionally, this architecture is well-suited to handle the diverse and multimodal nature of the "MusicCaps" dataset.
#

In [ ]:
import pandas as pd
from moviepy.video.io.VideoFileClip import VideoFileClip
from sklearn.model_selection import train_test_split
import os
import subprocess
import re
import re
from moviepy.video.io.VideoFileClip import VideoFileClip
from torch.nn.utils.rnn import pad_sequence

# 1. Read the CSV file
csv_file = "musiccaps-public.csv"
data = pd.read_csv(csv_file)

# Ensure necessary columns are present
if not {"ytid", "start_s", "end_s"}.issubset(data.columns):
    raise ValueError("CSV file must contain 'ytid', 'start_s', and 'end_s' columns.")

# Create directories for storing videos and clips
os.makedirs("videos", exist_ok=True)
os.makedirs("clips", exist_ok=True)

# 2. Download YouTube videos using yt-dlp
def download_video(ytid, output_dir="videos"):
    """Download a YouTube video using yt-dlp."""
    url = f"https://www.youtube.com/watch?v={ytid}"
    output_path = os.path.join(output_dir, f"{ytid}.mp4")
    try:
        subprocess.run(
            [
                "yt-dlp",
                "-f",
                "bestvideo[ext=mp4]+bestaudio[ext=m4a]/mp4",
                "--merge-output-format", "mp4",
                "-o", output_path,
                url,
            ],
            check=True,
        )
        return output_path
    except subprocess.CalledProcessError as e:
        print(f"Failed to download video {ytid}: {e}")
        return None


def sanitize_filename(filename):
    """Remove special characters from filenames."""
    return re.sub(r'[^\w\-_\. ]', '_', filename)

def extract_clip(video_path, start_s, end_s, output_dir="clips"):
    """Extract a clip from a video."""
    # Sanitize video file name
    sanitized_video_path = sanitize_filename(video_path)
    sanitized_video_path = os.path.basename(sanitized_video_path)

    # Ensure absolute paths
    video_path = os.path.abspath(video_path)
    output_dir = os.path.abspath(output_dir)

    clip_name = os.path.join(output_dir, f"{sanitized_video_path}_{start_s}_{end_s}.mp4")

    try:
        # Extract video clip
        clip = VideoFileClip(video_path).subclip(start_s, end_s)
        clip.write_videofile(clip_name, codec="libx264", audio_codec="aac")
    except Exception as e:
        print(f"Error processing clip from {video_path}: {e}")
        return None

    return clip_name

# Download and process all videos
clip_paths = []
for index, row in data.iterrows():
    ytid = row["ytid"]
    start_s = row["start_s"]
    end_s = row["end_s"]

    try:
        # Download video
        video_path = download_video(ytid)
        if video_path:  # Proceed only if the video was successfully downloaded
            # Extract clip
            clip_path = extract_clip(video_path, start_s, end_s)
            clip_paths.append(clip_path)
    except Exception as e:
        print(f"Failed to process YouTube ID {ytid}: {e}")

# 4. Split data into training, validation, and testing sets
clip_paths = pd.Series(clip_paths)  # Convert to pandas Series for easier handling
train_clips, test_clips = train_test_split(clip_paths, test_size=0.2, random_state=42)
train_clips, val_clips = train_test_split(train_clips, test_size=0.1, random_state=42)

print(f"Train Clips: {len(train_clips)}")
print(f"Validation Clips: {len(val_clips)}")
print(f"Test Clips: {len(test_clips)}")

# Save splits to disk (optional)
train_clips.to_csv("train_clips.csv", index=False)
val_clips.to_csv("val_clips.csv", index=False)
test_clips.to_csv("test_clips.csv", index=False)

In [ ]:
import os
from moviepy.editor import AudioFileClip

# Paths
input_dir = r"C:\Coding\MLProject\clips"
output_dir = r"C:\Coding\MLProject\wav_files"

# Create output directory if it doesn't exist
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# List all mp4 files in the input directory
mp4_files = sorted([f for f in os.listdir(input_dir) if f.endswith('.mp4')])

# Convert each file to .wav
for i, mp4_file in enumerate(mp4_files, start=1):
    input_path = os.path.join(input_dir, mp4_file)
    output_path = os.path.join(output_dir, f"{i}.wav")

    try:
        # Extract audio and save as .wav
        with AudioFileClip(input_path) as audio:
            audio.write_audiofile(output_path, codec="pcm_s16le")
        print(f"Converted: {mp4_file} -> {output_path}")
    except Exception as e:
        print(f"Failed to convert {mp4_file}: {e}")

print("Conversion complete.")


### Process the Audio


In [ ]:
from pydub import AudioSegment
import os

def convert_audio(file_path, output_format='wav'):
    """
    Converts an audio file to the specified format. Automatically detects the file type.

    Args:
    - file_path (str): The path to the audio file to convert.
    - output_format (str): The format to convert to (e.g., 'mp3', 'wav').

    Returns:
    - str: Path to the converted audio file.
    """
    try:
        # Check if the file exists
        if not os.path.exists(file_path):
            raise FileNotFoundError(f"File {file_path} not found.")

        # Load the audio file
        audio = AudioSegment.from_file(file_path)

        # Define the output file path
        file_name, _ = os.path.splitext(file_path)
        output_path = f"{file_name}.{output_format}"

        # Export the audio in the desired format
        audio.export(output_path, format=output_format)

        print(f"Conversion successful: {output_path}")
        return output_path
    except Exception as e:
        print(f"Error during conversion: {e}")
        return None

# Example Usage:
# converted_file = convert_audio('path/to/your/audio.mp3', 'wav')
# print(converted_file)

### MODEL 1 SPECTROGRAM FEATURES

In [ ]:
import os
import librosa
import numpy as np
from sklearn.model_selection import train_test_split

# Define the spectrogram conversion function
def convert_to_stft_spectrograms(file_path, interval=0.1, sr=22050, n_fft=2048, hop_length=None):
    """
    Converts a .wav audio file to normalized STFT spectrograms at specified intervals
    and stacks stereo channels into a 3D array.
    """
    y, sr = librosa.load(file_path, sr=sr, mono=False)
    if y.ndim == 1:
        y = np.vstack([y, y])  # Convert mono to pseudo-stereo

    if hop_length is None:
        hop_length = n_fft // 4

    # Update to calculate frame length for the given interval (0.1 seconds)
    frame_length = int(sr * interval)
    spectrograms = []

    for start in range(0, y.shape[-1], frame_length):
        end = start + frame_length
        if end > y.shape[-1]:
            break
        y_segment = y[:, start:end]
        stft_left = librosa.stft(y_segment[0], n_fft=n_fft, hop_length=hop_length)
        stft_right = librosa.stft(y_segment[1], n_fft=n_fft, hop_length=hop_length)
        mag_stft_left = np.abs(stft_left)
        mag_stft_right = np.abs(stft_right)
        stacked_spectrogram = np.dstack((mag_stft_left, mag_stft_right))

        # Normalize the spectrogram to range [0, 1], with a check to avoid division by zero
        epsilon = 1e-8  # A small value to avoid division by zero
        min_val = np.min(stacked_spectrogram)
        max_val = np.max(stacked_spectrogram)
        normalized_spectrogram = (stacked_spectrogram - min_val) / (max_val - min_val + epsilon)

        spectrograms.append(normalized_spectrogram)

    return np.array(spectrograms)


# Paths
wav_folder = r"C:\Coding\MLProject\wav_files"
train_folder = r"C:\Coding\MLProject\training_spectrograms"
test_folder = r"C:\Coding\MLProject\testing_spectrograms"

# Create directories if they don't exist
os.makedirs(train_folder, exist_ok=True)
os.makedirs(test_folder, exist_ok=True)

# Process each .wav file
spectrogram_files = []
for file in os.listdir(wav_folder):
    if file.endswith(".wav"):
        file_path = os.path.join(wav_folder, file)
        try:
            spectrograms = convert_to_stft_spectrograms(file_path, interval=0.1)  # 0.1-second interval
            spectrogram_files.append((file, spectrograms))
        except Exception as e:
            print(f"Error processing {file_path}: {e}")

# Split into training and testing
train_files, test_files = train_test_split(spectrogram_files, test_size=0.2, random_state=42)

# Save spectrograms
def save_spectrograms(file_list, output_folder):
    for file_name, spectrograms in file_list:
        base_name = os.path.splitext(file_name)[0]
        for i, spectrogram in enumerate(spectrograms):
            save_path = os.path.join(output_folder, f"{base_name}_spectrogram_{i}.npy")
            np.save(save_path, spectrogram)

save_spectrograms(train_files, train_folder)
save_spectrograms(test_files, test_folder)

print(f"Processed {len(train_files)} training files and {len(test_files)} testing files.")


In [ ]:
def normalize_spectrograms(spectrograms):
    """
    Normalize the spectrograms to have zero mean and unit variance.

    Parameters:
    - spectrograms (numpy array): 3D array of spectrograms of shape (num_frames, n_freq_bins, n_channels).

    Returns:
    - normalized_spectrograms (numpy array): Normalized 3D array.
    """
    # Normalize across the frequency bins and time frames (per channel)
    spectrograms = np.log1p(spectrograms)  # Apply log scaling
    mean = np.mean(spectrograms, axis=(0, 1), keepdims=True)
    std = np.std(spectrograms, axis=(0, 1), keepdims=True)
    normalized_spectrograms = (spectrograms - mean) / std  # Normalize by mean and std
    return normalized_spectrograms



In [ ]:
from sklearn.model_selection import train_test_split

def prepare_data(spectrograms, test_size=0.2, validation_size=0.1):
    """
    Prepare data for model training by splitting into train, validation, and test sets.
    For unsupervised tasks, labels are not required.

    Parameters:
    - spectrograms (numpy array): Normalized spectrograms.
    - test_size (float): Proportion of the data to use for testing.
    - validation_size (float): Proportion of the training data to use for validation.

    Returns:
    - X_train, X_val, X_test: Train, validation, and test splits.
    """
    # Split into train + test
    X_train, X_test = train_test_split(spectrograms, test_size=test_size, random_state=42)

    # Split the training data further into train + validation
    X_train, X_val = train_test_split(X_train, test_size=validation_size, random_state=42)

    return X_train, X_val, X_test


In [ ]:
import os
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.nn.functional as F

# Custom Dataset
class SpectrogramDataset(Dataset):
    import os
    import numpy as np
    import torch
    from torch.utils.data import Dataset, DataLoader
    import torch.nn as nn
    import torch.nn.functional as F
    def __init__(self, directory):
        self.files = [os.path.join(directory, f) for f in os.listdir(directory) if f.endswith('.npy')]

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        # Load spectrogram as numpy array and convert to tensor
        spectrogram = np.load(self.files[idx])  # Shape: (1025, 5, 2)
        spectrogram = torch.tensor(spectrogram, dtype=torch.float32)
        spectrogram = spectrogram.permute(2, 0, 1)  # Change shape to (2, 1025, 5)

        # Resize spectrogram to (2, 16, 16) using interpolation
        spectrogram = F.interpolate(spectrogram.unsqueeze(0), size=(16, 16), mode='bilinear', align_corners=False).squeeze(0)

        return spectrogram


# CNN Model
class SpectrogramCNN(nn.Module):
    def __init__(self, input_channels=2):
        super(SpectrogramCNN, self).__init__()
        # **Encoder**
        self.encoder = nn.Sequential(
        nn.Conv2d(input_channels, 32, kernel_size=3, stride=1, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2, stride=2),
        nn.Dropout(0.3),  # Dropout with 30% probability

        nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2, stride=2),
        nn.Dropout(0.3),

        nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
        nn.ReLU(),
        nn.AdaptiveAvgPool2d((4, 4))
)


        # **Decoder**
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(128, 64, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),

            nn.ConvTranspose2d(64, 32, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),

            nn.ConvTranspose2d(32, input_channels, kernel_size=3, stride=1, padding=1)
)


    def forward(self, x):
        x = self.encoder(x)
        x = self.decoder(x)
        return x


# Training Function
from tqdm import tqdm

def train_model(model, train_loader, val_loader, epochs=10, lr=0.0001, patience=10, device = "cpu"):
    from torch.optim import AdamW
    criterion = nn.MSELoss()  # Use Mean Squared Error for reconstruction
    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=1e-4)

    model.to(device)

    best_val_loss = float('inf')  # Initialize the best validation loss
    best_model = None  # Variable to save the best model
    patience_counter = 0  # Counter for early stopping
    train_losses = []
    val_losses = []

    for epoch in range(epochs):
        model.train()
        train_loss = 0

        # Progress bar for training loop
        for spectrograms in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} - Training", unit="batch", leave=False):
            spectrograms = spectrograms.to(device)

            optimizer.zero_grad()
            outputs = model(spectrograms)

            loss = criterion(outputs, spectrograms)  # Compare the reconstructed input with the original input
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        train_loss /= len(train_loader)
        train_losses.append(train_loss)  # Log training loss


        val_loss = 0
        model.eval()

        # Progress bar for validation loop
        with torch.no_grad():
            for spectrograms in tqdm(val_loader, desc="Validation", unit="batch", leave=False):
                spectrograms = spectrograms.to(device)
                outputs = model(spectrograms)

                loss = criterion(outputs, spectrograms)
                val_loss += loss.item()


        val_loss /= len(val_loader)
        val_losses.append(val_loss)  # Log validation loss

        print(f"\nEpoch {epoch+1}/{epochs}, Train Loss: {train_loss:.4f}, Validation Loss: {val_loss:.4f}")

        # Early stopping logic
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model = model.state_dict()  # Save model state dictionary
            patience_counter = 0  # Reset counter if validation loss improves
            print(f"Best model saved at epoch {epoch+1} with Validation Loss: {val_loss:.4f}")
        else:
            patience_counter += 1  # Increment counter if no improvement
            print(f"No improvement in Validation Loss for {patience_counter} epochs.")

        if patience_counter >= patience:
            print(f"Early stopping triggered at epoch {epoch+1}. Best Validation Loss: {best_val_loss:.4f}")
            break

    return best_model  # Return the state dictionary of the best model



In [ ]:
# Paths
train_dir = 'training_spectrograms'
test_dir = 'testing_spectrograms'

# Dataset and Dataloader
train_dataset = SpectrogramDataset(train_dir)
val_dataset = SpectrogramDataset(test_dir)

# Save datasets to disk
torch.save(train_dataset, 'train_dataset.pt')
torch.save(val_dataset, 'val_dataset.pt')

print('Datasets have been cached.')


In [ ]:
train_dataset = torch.load('train_dataset.pt', weights_only = False)
val_dataset = torch.load('val_dataset.pt', weights_only = False)

train_loader = DataLoader(train_dataset, batch_size=25, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=25, shuffle=False)

print('Datasets loaded from cache.')
# Model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SpectrogramCNN()
print(device)

# Train the model and save the best model
best_model = train_model(model, train_loader, val_loader, epochs=100, lr=0.001, device=device)

# Load best model weights back into the model
model.load_state_dict(best_model)
model.to(device)
model.eval()  # Set the model to evaluation mode

print('Best model has been loaded back into memory.')
torch.save(best_model, 'model_1.pth')
print("Best model saved for future use")



###MODEL 2


Objective: Given spectrograms of shape (2, 16, 16), predict an aspect list. This is a multi-label classification problem where:
	•	Input: Spectrograms of shape (2, 16, 16).
	•	Output: A binary vector of length 11349 (one-hot encoded), where 1 indicates the presence of an aspect.


In [3]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from torch.utils.data import Dataset, DataLoader
import torch

# Paths
csv_file = "musiccaps-public.csv"  # Replace with the correct path
spectrogram_dir = "resources/spectrograms"

# Step 1: Load the CSV file
df = pd.read_csv(csv_file)

# Process the 'aspect_list' column into a list of aspects
df['aspects'] = df['aspect_list'].apply(lambda x: x.strip("[]").replace("'", "").split(", "))

# Step 2: MultiLabel Binarizer to create fixed-size binary vectors
all_aspects = set(aspect for aspects in df['aspects'] for aspect in aspects)
mlb = MultiLabelBinarizer(classes=sorted(all_aspects))  # Fixed vocabulary order
aspect_binary = mlb.fit_transform(df['aspects'])  # Shape: (num_samples, n_labels)

# Save processed binary aspect labels
np.save("resources/aspect_labels.npy", aspect_binary)

# Step 3: Match spectrogram files to CSV entries
spectrogram_files = sorted([f for f in os.listdir(spectrogram_dir) if f.endswith('.npy')])

# Group spectrograms by their base ID (1, 2, ..., corresponding to DataFrame rows)
file_groups = {}
for f in spectrogram_files:
    base_id, spectrogram_idx = f.split('_spectrogram_')
    if base_id not in file_groups:
        file_groups[base_id] = []
    file_groups[base_id].append(os.path.join(spectrogram_dir, f))

# Ensure spectrograms are sorted within each group
for base_id in file_groups:
    file_groups[base_id] = sorted(file_groups[base_id], key=lambda x: int(x.split('_')[-1].split('.')[0]))

# Step 4: Filter out missing spectrograms
num_rows = len(df)
valid_ids = set(int(base_id) for base_id in file_groups.keys())
valid_indices = [idx for idx in range(num_rows) if (idx + 1) in valid_ids]

# Print missing IDs for verification
missing_ids = [i + 1 for i in range(num_rows) if i + 1 not in valid_ids]
if missing_ids:
    print(f"Warning: Missing spectrograms for IDs: {missing_ids}")

# Retain only valid rows with corresponding spectrograms
df = df.loc[valid_indices].reset_index(drop=True)
aspect_binary = aspect_binary[valid_indices]

# Updated file_groups to map DataFrame row indices
row_to_spectrograms = {idx: file_groups[str(idx + 1)] for idx in range(len(df))}

# Step 5: Train, validation, and test split
indices = list(range(len(df)))
train_idx, test_idx = train_test_split(indices, test_size=0.2, random_state=42)
train_idx, val_idx = train_test_split(train_idx, test_size=0.1, random_state=42)

print(f"Train samples: {len(train_idx)}, Validation samples: {len(val_idx)}, Test samples: {len(test_idx)}")

# Step 6: Dataset Class
class SpectrogramAspectDataset(Dataset):
    def __init__(self, spectrogram_groups, aspect_labels, indices):
        """
        Args:
            spectrogram_groups (dict): Mapping of row indices to spectrogram file paths (lists).
            aspect_labels (ndarray): Binary aspect label matrix.
            indices (list): Subset indices for train, val, or test split.
        """
        # Filter indices to include only existing spectrograms
        self.valid_indices = [i for i in indices if i in spectrogram_groups]
        self.aspect_labels = aspect_labels[self.valid_indices]
        self.file_paths = [spectrogram_groups[i] for i in self.valid_indices]

        print(f"Dataset initialized with {len(self.valid_indices)} valid samples.")

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        # Load all spectrogram files (100 spectrograms for this row)
        spectrogram_files = self.file_paths[idx]
        spectrograms = [np.load(f) for f in spectrogram_files]  # List of arrays with shape (2, 16, 16)
        spectrograms = torch.tensor(np.stack(spectrograms), dtype=torch.float32)  # Shape: (100, 2, 16, 16)

        # Load corresponding aspect label
        label = torch.tensor(self.aspect_labels[idx], dtype=torch.float32)  # Binary vector

        return spectrograms, label

# Step 7: DataLoaders
train_dataset = SpectrogramAspectDataset(row_to_spectrograms, aspect_binary, train_idx)
val_dataset = SpectrogramAspectDataset(row_to_spectrograms, aspect_binary, val_idx)
test_dataset = SpectrogramAspectDataset(row_to_spectrograms, aspect_binary, test_idx)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)

# Step 8: Verify DataLoaders
print("DataLoader verification:")
for batch_idx, (spectrograms, labels) in enumerate(train_loader):
    print(f"Batch {batch_idx}: Spectrograms shape {spectrograms.shape}, Labels shape {labels.shape}")
    break



Train samples: 3848, Validation samples: 428, Test samples: 1070
Dataset initialized with 3848 valid samples.
Dataset initialized with 428 valid samples.
Dataset initialized with 1070 valid samples.
DataLoader verification:
Batch 0: Spectrograms shape torch.Size([8, 100, 1025, 5, 2]), Labels shape torch.Size([8, 13219])


In [50]:
import torch
import torch.nn as nn

class SpectrogramAspectClassifier(nn.Module):
    def __init__(self, num_aspects):
        super(SpectrogramAspectClassifier, self).__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(2, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),  # Pool reduces height and width by half

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)
        )

        self.fc_layers = nn.Sequential(
            nn.Linear(2048000, 512),  # Ensure fully connected layer dimensions match flattened sizes
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_aspects),
            nn.Sigmoid()
        )

    def forward(self, x):
        #print(f"Input shape: {x.shape}")

        # Step 1: Merge `[100, 5]` dimensions into a single batch dimension
        x = x.view(x.shape[0], x.shape[1] * x.shape[3], x.shape[2], x.shape[4])
        #print(f"After reshaping: {x.shape}")

        # Step 2: Permute to channels-first format required by Conv2d
        x = x.permute(0, 3, 1, 2)
        #print(f"After permutation: {x.shape}")

        # Step 3: Apply convolutional transformations
        x = self.conv_layers(x)
        #print(f"After conv_layers: {x.shape}")

        # Step 4: Flatten to prepare fully connected input
        x = x.contiguous().view(x.size(0), -1)
        #print(f"After flattening fully connected input: {x.shape}")

        x = self.fc_layers(x)
        return x






from tqdm import tqdm

# Training
def train_model(model, train_loader, val_loader, num_epochs=10, lr=0.001, device="cuda"):
    model = model.to(device)
    criterion = nn.BCELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0.0

        # Use tqdm to show progress bar for training loop
        train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} Training", unit="batch")
        for spectrograms, labels in train_bar:
            spectrograms, labels = spectrograms.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(spectrograms)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

            train_bar.set_postfix({"loss": loss.item()})

        # Evaluate on the validation set
        val_loss, val_f1 = evaluate_model(model, val_loader, device)
        print(f"Epoch {epoch+1}/{num_epochs} | "
              f"Train Loss: {total_loss/len(train_loader):.4f} | "
              f"Val Loss: {val_loss:.4f} | Val F1: {val_f1:.4f}")


def evaluate_model(model, loader, device):
    model.eval()
    val_loss = 0.0
    outputs_all, labels_all = [], []

    # Use tqdm to show progress bar for validation loop
    val_bar = tqdm(loader, desc="Evaluating", unit="batch")
    with torch.no_grad():
        for spectrograms, labels in val_bar:
            spectrograms, labels = spectrograms.to(device), labels.to(device)
            outputs = model(spectrograms)
            loss = nn.BCELoss()(outputs, labels)
            val_loss += loss.item()

            outputs_all.append(outputs.cpu())
            labels_all.append(labels.cpu())

            val_bar.set_postfix({"batch_loss": loss.item()})

    outputs_all = torch.cat(outputs_all).numpy()
    labels_all = torch.cat(labels_all).numpy()
    f1 = f1_score((outputs_all > 0.5).astype(int), labels_all, average="micro")
    return val_loss / len(loader), f1



In [ ]:

# Data Setup
aspect_labels = np.load("resources/aspect_labels.npy")
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# Train the model
device = "cuda" if torch.cuda.is_available() else "cpu"
model_2 = SpectrogramAspectClassifier(num_aspects=aspect_labels.shape[1])
train_model(model, train_loader, val_loader, num_epochs=10, lr=0.001, device=device)


torch.save(model_2, 'model_2.pth')
print("Best model saved for future use")


Epoch 1/10 Training:  33%|███▎      | 158/481 [18:07<38:49,  7.21s/batch, loss=0.00586]

In [ ]:
#Example code for later:
#
# model = SpectrogramCNN(input_channels=2)  # Ensure architecture matches
# model.load_state_dict(torch.load('best_model.pth'))
# model.eval()  # Set to evaluation mode

#
# class LargerModel(nn.Module):
#     def __init__(self, existing_model, transformer_model):
#         super(LargerModel, self).__init__()
#         self.existing_model = existing_model  # Your loaded model
#         self.transformer = transformer_model  # Your transformer-based model
#
#     def forward(self, x):
#         x = self.existing_model(x)
#         x = self.transformer(x)
#         return x
#
# # Example usage
# transformer_model = SomeTransformerModel()  # Replace with your transformer
# larger_model = LargerModel(model, transformer_model)
#


###MODEL 3 CAPTION GENERATOR

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
import torch
import torch.nn as nn
from transformers import BertTokenizer, BertModel

# Load dataset
data = pd.read_csv('musiccaps-public.csv')

# We will retain only relevant columns: caption and aspect_list
data = data[['caption', 'aspect_list']]

# Convert `aspect_list` to a list of meaningful audio classes
data['aspect_list'] = data['aspect_list'].apply(lambda x: x.split(','))

# Split the dataset into training and validation sets (80% train, 20% validation)
train_data, val_data = train_test_split(data, test_size=0.2, random_state=42)

print(f"Number of training samples: {len(train_data)}")
print(f"Number of validation samples: {len(val_data)}")

# Prepare the tokenizer
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

class MusicDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length=128):
        self.captions = dataframe['caption'].tolist()
        self.labels = dataframe['aspect_list'].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

        # Create a consistent list of all classes (aspects)
        self.all_classes = sorted(set(label for labels in dataframe['aspect_list'] for label in labels))

    def __getitem__(self, index):
        # Get all the labels for an item and convert them into a multi-hot vector
        labels = self.labels[index]
        multi_hot = torch.zeros(len(self.all_classes))
        for label in labels:
            if label in self.all_classes:
                multi_hot[self.all_classes.index(label)] = 1

        # Tokenize the caption (which is the target)
        caption = self.captions[index]
        encoding = self.tokenizer.encode_plus(
            caption,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )

        target_input_ids = encoding['input_ids'].squeeze()
        attention_mask = encoding['attention_mask'].squeeze()

        return {
            'input_ids': multi_hot,
            'target_input_ids' : target_input_ids,
            'attention_mask': attention_mask,
        }

    def __len__(self):
        return len(self.captions)


# Prepare the dataset and DataLoader
def collate_fn(batch):
    inputs = torch.stack([item['input_ids'] for item in batch])
    target_input_ids = torch.stack([item['target_input_ids'] for item in batch])
    attention_masks = torch.stack([item['attention_mask'] for item in batch])

    return {
        'inputs': inputs,
        'target_input_ids': target_input_ids,
        'attention_masks': attention_masks
    }

train_dataset = MusicDataset(train_data, tokenizer)
train_loader = DataLoader(train_dataset, batch_size=25, shuffle=True, collate_fn=collate_fn)

val_dataset = MusicDataset(val_data, tokenizer)
val_loader = DataLoader(val_dataset, batch_size=25, shuffle=False, collate_fn=collate_fn)

import torch
import torch.nn.functional as F

class AttentionLayer(torch.nn.Module):
    def __init__(self, embedding_dim):
        super(AttentionLayer, self).__init__()
        self.embedding_dim = embedding_dim
        # Linear transformations for query (aspect embeddings), key, and value (BERT hidden states)
        self.query_linear = torch.nn.Linear(embedding_dim, embedding_dim)
        self.key_linear = torch.nn.Linear(embedding_dim, embedding_dim)
        self.value_linear = torch.nn.Linear(embedding_dim, embedding_dim)
        self.attention_dropout = torch.nn.Dropout(0.1)

    def forward(self, embedded_aspects, bert_hidden_states):
        # embedded_aspects: (batch_size, num_aspects, embedding_dim)
        # bert_hidden_states: (batch_size, seq_len, embedding_dim)

        batch_size, num_aspects, _ = embedded_aspects.shape
        seq_len = bert_hidden_states.size(1)

        # Linear transformations to obtain queries, keys, and values
        queries = self.query_linear(embedded_aspects)  # (batch_size, num_aspects, embedding_dim)
        keys = self.key_linear(bert_hidden_states)  # (batch_size, seq_len, embedding_dim)
        values = self.value_linear(bert_hidden_states)  # (batch_size, seq_len, embedding_dim)

        # Compute attention scores (dot product between queries and keys)
        attention_scores = torch.matmul(queries, keys.transpose(-2, -1))  # (batch_size, num_aspects, seq_len)
        attention_scores = attention_scores / (self.embedding_dim ** 0.5)  # Scale by sqrt(embedding_dim)

        # Apply softmax to get attention weights
        attention_weights = F.softmax(attention_scores, dim=-1)  # (batch_size, num_aspects, seq_len)

        # Optionally apply dropout
        attention_weights = self.attention_dropout(attention_weights)

        # Compute weighted sum of values (BERT hidden states)
        attended_values = torch.matmul(attention_weights, values)  # (batch_size, num_aspects, embedding_dim)

        return attended_values

class CaptionGenerator(nn.Module):
    def __init__(self, num_aspects, vocab_size, max_length):
        super(CaptionGenerator, self).__init__()
        self.embedding_dim = 128
        self.hidden_dim = 256
        self.max_length = max_length

        # Input layer for multi-hot aspect list
        self.aspect_fc = nn.Linear(self.embedding_dim, 768)  # Match output to BERT hidden size (768)

        # BERT model setup
        self.bert = BertModel.from_pretrained("bert-base-uncased")

        # Attention layer
        self.attention_layer = AttentionLayer(embedding_dim=768)  # Match BERT's hidden size

        # Adjust LSTM input size to match BERT's output (768)
        self.lstm = nn.LSTM(input_size=768, hidden_size=self.hidden_dim, num_layers=2, batch_first=True)

        # Decoder for generating captions
        self.embedding = nn.Embedding(vocab_size, self.embedding_dim)
        self.fc_out = nn.Linear(self.hidden_dim, vocab_size)

    def forward(self, inputs, target_input_ids, attention_mask):
        inputs = inputs.long().to(device)
        target_input_ids = target_input_ids.long().to(device)

        # Embed aspects (multi-hot encoding)
        embedded_aspects = self.embedding(inputs)  # (batch_size, num_aspects, embedding_dim)
        embedded_aspects = embedded_aspects.view(-1, self.embedding_dim)
        embedded_aspects = self.aspect_fc(embedded_aspects)  # (batch_size * num_aspects, 768)
        embedded_aspects = embedded_aspects.view(inputs.size(0), -1, 768)  # (batch_size, num_aspects, 768)

        # Pass through BERT
        bert_output = self.bert(input_ids=target_input_ids, attention_mask=attention_mask)
        bert_hidden_states = bert_output.last_hidden_state  # (batch_size, seq_len, 768)

        # Apply attention
        attended_values = self.attention_layer(embedded_aspects, bert_hidden_states)  # (batch_size, num_aspects, 768)

        # Mean pooling to combine attended values
        expanded_attended_values = attended_values.mean(dim=1)  # (batch_size, 768)
        expanded_attended_values = expanded_attended_values.unsqueeze(1).repeat(1, self.max_length, 1)  # (batch_size, seq_len, 768)

        # Pass through LSTM
        lstm_out, _ = self.lstm(expanded_attended_values)  # (batch_size, seq_len, hidden_dim)
        logits = self.fc_out(lstm_out)  # (batch_size, seq_len, vocab_size)

        # Reshape for loss computation
        outputs = logits.view(-1, vocab_size)  # (batch_size * seq_len, vocab_size)
        return outputs



from tqdm.autonotebook import tqdm as notebook_tqdm

from tqdm import tqdm

def train_caption_model(model, train_loader, val_loader, optimizer, criterion, epochs):
    for epoch in range(epochs):
        model.train()
        train_loss = 0

        # Add tqdm progress bar for training
        for batch in tqdm(train_loader, desc=f"Epoch {epoch + 1}/{epochs} - Training", leave=False):
            # Unpack batch dictionary
            aspects = batch['inputs'].to(device)  # Multi-hot encoded inputs
            target_input_ids = batch['target_input_ids'].to(device)
            attention_masks = batch['attention_masks'].to(device)

            # Forward pass
            outputs = model(aspects, target_input_ids, attention_masks)

            # Reshape outputs and targets
            outputs = outputs.view(-1, vocab_size)  # Shape: [batch_size * seq_len, vocab_size]
            target_input_ids = target_input_ids.view(-1)  # Shape: [batch_size * seq_len]

            # Compute loss
            loss = criterion(outputs, target_input_ids)  # `ignore_index` handles padding tokens

            # Backward pass and optimization
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        # Validation loop
        model.eval()
        val_loss = 0
        with torch.no_grad():
            # Add tqdm progress bar for validation
            for batch in tqdm(val_loader, desc=f"Epoch {epoch + 1}/{epochs} - Validation", leave=False):
                aspects = batch['inputs'].to(device)
                target_input_ids = batch['target_input_ids'].to(device)
                attention_masks = batch['attention_masks'].to(device)

                outputs = model(aspects, target_input_ids, attention_masks)
                outputs = outputs.view(-1, vocab_size)
                target_input_ids = target_input_ids.view(-1)
                loss = criterion(outputs, target_input_ids)

                val_loss += loss.item()

        # Print epoch losses
        print(f"Epoch {epoch + 1}/{epochs}, Training Loss: {train_loss / len(train_loader):.4f}, Validation Loss: {val_loss / len(val_loader):.4f}")


# Hyperparameters
device = torch.device("cpu")
vocab_size = tokenizer.vocab_size
max_length = 128
num_aspects = len(train_dataset.all_classes)
criterion = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_token_id)


# Model, optimizer, and loss criterion
model = CaptionGenerator(num_aspects, vocab_size, max_length)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()  # Use CrossEntropyLoss for classification
print("Device:", device)
# Move model to device
model = model.to(device)

# Train the model
train_caption_model(model, train_loader, val_loader, optimizer, criterion, epochs=10)

# Save the model

torch.save(model.state_dict(), 'model_3.pth')
print("Model saved as 'model_3.pth'")


Number of training samples: 4416
Number of validation samples: 1105
Device: cpu


Epoch 1/10, Training Loss: 3.6425, Validation Loss: 3.1350


Epoch 2/10, Training Loss: 2.9235, Validation Loss: 2.9207


Epoch 3/10, Training Loss: 2.8479, Validation Loss: 2.8931


Epoch 4/10, Training Loss: 2.8226, Validation Loss: 2.9006


...

Epoch 6/10, Training Loss: 2.7962, Validation Loss: 2.8790


Epoch 7/10, Training Loss: 2.7900, Validation Loss: 2.8739


Epoch 8/10, Training Loss: 2.7835, Validation Loss: 2.8560


Epoch 9/10, Training Loss: 2.7860, Validation Loss: 2.8635


Epoch 10/10, Training Loss: 2.7785, Validation Loss: 2.8535
Model saved as 'caption_generator_bert.pth'


### MODEL 4 ASPECT LIST ALIGMENT MODEL

In [ ]:
import torch
import torch.nn as nn

class SpectrogramAspectClassifier(nn.Module):
    def __init__(self, caption_embedding_size=768):  # Default for BERT
        super(SpectrogramAspectClassifier, self).__init__()

        self.conv1 = nn.Conv2d(in_channels=2, out_channels=64, kernel_size=3, stride=1, padding=1)
        self.bn1 = nn.BatchNorm2d(64)
        self.conv2 = nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1)
        self.bn2 = nn.BatchNorm2d(128)
        self.conv3 = nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=1)
        self.bn3 = nn.BatchNorm2d(256)

        # Updated caption embedding to handle dynamic input size
        self.caption_embedding = nn.Linear(caption_embedding_size, 512)  # Adjust input and output dimensions

        # Define adaptive pooling layers
        self.adaptive_pool1 = nn.AdaptiveAvgPool2d((8, 8))
        self.adaptive_pool2 = nn.AdaptiveAvgPool2d((4, 4))
        self.adaptive_pool3 = nn.AdaptiveAvgPool2d((2, 2))

        # Fully connected layer for combined input
        self.fc = nn.Linear(256 * 2 * 2 + 512, 13219)  # Adjusted for fixed pooling size

    def forward(self, spectrogram, captions):
        # Permute spectrogram dimensions (adjust based on input format)
        spectrogram = spectrogram.permute(0, 3, 1, 2)

        # First convolution block
        x = torch.relu(self.bn1(self.conv1(spectrogram)))
        x = self.adaptive_pool1(x)  # Apply Adaptive Pooling
        x = torch.relu(self.bn2(self.conv2(x)))
        x = self.adaptive_pool2(x)  # Apply Adaptive Pooling
        x = torch.relu(self.bn3(self.conv3(x)))
        x = self.adaptive_pool3(x)  # Apply Adaptive Pooling

        x = x.reshape(x.size(0), -1)  # Flatten for fully connected layers

        # Process captions
        captions_embedded = self.caption_embedding(captions)  # Embed captions

        # Concatenate flattened spectrogram `x` and `captions_embedded`
        combined = torch.cat((x, captions_embedded), dim=1)

        # Output layer (raw logits for BCEWithLogitsLoss)
        output = self.fc(combined)
        return output

# Use BCEWithLogitsLoss for multi-label classification
criterion = torch.nn.BCEWithLogitsLoss()



In [ ]:
from tqdm import tqdm
from sklearn.metrics import accuracy_score, f1_score

def train_aspect_classifier(model, train_loader, val_loader, optimizer, criterion, num_epochs, device):
    model.to(device)

    for epoch in range(num_epochs):
        model.train()
        train_loss = 0.0

        # Training phase
        for batch_idx, batch in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} - Training")):
            try:
                # Retrieve inputs from the batch
                spectrograms = batch['spectrogram'].to(device)
                captions = batch['caption'].to(device)
                aspects = batch['aspects'].to(device)

                # Ensure proper data types
                spectrograms = spectrograms.float()
                captions = captions.long()
                aspects = aspects.float()

                # Zero the parameter gradients
                optimizer.zero_grad()

                # Forward pass
                outputs = model(spectrograms, captions)

                # Compute loss
                loss = criterion(outputs, aspects)
                train_loss += loss.item()

                # Backward pass and optimization
                loss.backward()
                optimizer.step()

            except Exception as e:
                print(f"[ERROR] Issue in training batch {batch_idx}: {e}")
                continue

        avg_train_loss = train_loss / len(train_loader)
        print(f"Epoch {epoch+1}/{num_epochs} - Training Loss: {avg_train_loss:.4f}")

        # Validation phase
        evaluate_aspect_classifier(model, val_loader, device)

def evaluate_aspect_classifier(model, data_loader, device):
    model.eval()
    all_preds, all_labels = [], []
    val_loss = 0.0

    with torch.no_grad():
        for batch_idx, batch in enumerate(tqdm(data_loader, desc="Validating")):
            try:
                spectrograms = batch['spectrogram'].to(device)
                captions = batch['caption'].to(device)
                aspects = batch['aspects'].to(device)

                spectrograms = spectrograms.float()
                captions = captions.long()
                aspects = aspects.float()

                # Forward pass
                outputs = model(spectrograms, captions)

                # Compute loss
                loss = criterion(outputs, aspects)
                val_loss += loss.item()

                # Collect predictions and true labels for metrics
                predictions = (outputs > 0.5).float()
                all_preds.append(predictions.cpu().numpy())
                all_labels.append(aspects.cpu().numpy())

            except Exception as e:
                print(f"[ERROR] Issue in validation batch {batch_idx}: {e}")
                continue

    all_preds = np.concatenate(all_preds, axis=0)
    all_labels = np.concatenate(all_labels, axis=0)

    # Calculate metrics
    accuracy = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average="micro")

    avg_val_loss = val_loss / len(data_loader)
    print(f"Validation Loss: {avg_val_loss:.4f}, Accuracy: {accuracy:.4f}, F1 Score: {f1:.4f}")


In [ ]:
import pandas as pd

# Load the CSV file containing your dataset
data = pd.read_csv('musiccaps-public.csv')

# Get all unique aspects from the dataset
all_aspects = set()
for aspect_list in data['aspect_list']:
    aspects = eval(aspect_list)  # Convert the string representation of list back to a Python list
    all_aspects.update(aspects)

# Create a mapping from aspect name to index
aspect_to_index = {aspect: idx for idx, aspect in enumerate(sorted(all_aspects))}
print(f"Total unique aspects: {len(aspect_to_index)}")

# Convert the aspect list from strings to indices
data['aspect_indices'] = data['aspect_list'].apply(lambda x: [aspect_to_index[aspect] for aspect in eval(x) if aspect in aspect_to_index])
print(data[['aspect_list', 'aspect_indices']].head())

from transformers import BertTokenizer

# Initialize BERT tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Tokenize captions
data['caption_tokens'] = data['caption'].apply(lambda x: tokenizer(x, padding="max_length", truncation=True, max_length=128, return_tensors="pt"))
print(data[['caption', 'caption_tokens']].head())



In [ ]:
from sklearn.model_selection import train_test_split

# Split into training and validation sets (80% train, 20% validation)
train_data, val_data = train_test_split(data, test_size=0.2, random_state=42)

# Optionally, split the validation set into validation and test sets (e.g., 50% validation, 50% test)
val_data, test_data = train_test_split(val_data, test_size=0.5, random_state=42)

# Check the splits
print(f"Training samples: {len(train_data)}")
print(f"Validation samples: {len(val_data)}")
print(f"Test samples: {len(test_data)}")


In [ ]:
import torch
def custom_collate_fn(batch):
    # Filter out any None items (from __getitem__ exceptions)
    batch = [item for item in batch if item is not None]

    if len(batch) == 0:
        return None  # Handle edge case where all items are None

    # Process spectrograms
    spectrograms = torch.stack([
        torch.tensor(item['spectrogram'], dtype=torch.float32)
        if not isinstance(item['spectrogram'], torch.Tensor) else item['spectrogram'].clone().detach().float()
        for item in batch
    ])

    # Process captions (already tensors)
    input_ids = [item['caption'] for item in batch]
    max_length = max([len(ids) for ids in input_ids])

    # Pad captions to the maximum length
    padded_captions = torch.zeros((len(input_ids), max_length), dtype=torch.long)
    for i, ids in enumerate(input_ids):
        padded_captions[i, :len(ids)] = ids

    # Process aspects
    num_aspects = 13219  # Adjust size for the number of aspects
    aspects_tensor = torch.zeros((len(batch), num_aspects), dtype=torch.float32)
    for idx, item in enumerate(batch):
        for aspect in item['aspects']:
            aspects_tensor[idx, aspect] = 1.0

    return {
        'spectrogram': spectrograms,
        'caption': padded_captions,
        'aspects': aspects_tensor
    }



In [ ]:
import os  # Add this import at the top of your script
import torch
from torch.utils.data import DataLoader
from transformers import BertTokenizer
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np
from dataset import MultimodalDataset

# Prepare the dataset
train_data, val_data = train_test_split(data, test_size=0.2, random_state=42)

# Create the train and validation datasets
train_dataset = MultimodalDataset(train_data, tokenizer, aspect_to_index)
val_dataset = MultimodalDataset(val_data, tokenizer, aspect_to_index)

# Create DataLoader
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    collate_fn=custom_collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=custom_collate_fn
)


# Model and training setup
device = torch.device("cpu")

# Initialize the aspect classifier model
model = SpectrogramAspectClassifier()  # Based on dataset info
model.to(device)

# Set up optimizer and loss function
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = torch.nn.BCELoss()  # Binary Cross-Entropy loss for multi-label classification

# Training loop
model_4 = train_aspect_classifier(model, train_loader, val_loader, optimizer, criterion, num_epochs=5, device=device)
torch.save(best_model, 'model_4.pth')
print("Best model saved for future use")



### MODEL 5 THE TRANSFORMER MODEL

In [ ]:
import os
import torch
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader
from transformers import BertTokenizer, BertModel
from dataset import MultimodalDataset
from torch.utils.data import Dataset

class DataHandler:
    def __init__(self, csv_file, spectrogram_val_dir, tokenizer, aspect_to_index, test_size = 0.2):
        self.csv_file = csv_file
        self.spectrogram_train_dir = spectrogram_train_dir
        self.spectrogram_val_dir = spectrogram_val_dir
        self.tokenizer = tokenizer
        self.aspect_to_index = aspect_to_index
        self.test_size = test_size
        self.train_data, self.val_data = self._load_and_split_data()

    def _load_and_split_data(self):
        data = pd.read_csv(self.csv_file)
        return train_test_split(data, test_size=self.test_size, random_state=42)
    # the above line is, for some reason, having test_size as something other than

    def create_datasets(self):
        train_dataset = MultimodalDataset( self.train_data, self.tokenizer, self.aspect_to_index, self.spectrogram_train_dir)
        # I added self.spectrogram_val_dir to the above line because it said it was missing it
        val_dataset = MultimodalDataset(self.val_data, self.tokenizer, self.aspect_to_index, self.spectrogram_val_dir)
        return train_dataset, val_dataset

    def create_dataloaders(self, train_dataset, val_dataset, batch_size=25, num_workers=4):
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers)
        val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)
        return train_loader, val_loader


class ModelHandler:
    def __init__(self, audio_model_class, text_model_class, device="cuda"):
        self.device = device
        self.audio_model_class = audio_model_class
        self.text_model_class = text_model_class

    def load_audio_model(self, model_path):
        audio_model = self.audio_model_class().to(self.device)
        audio_model.load_state_dict(torch.load(model_path, map_location=self.device))
        audio_model.eval()
        return audio_model

    def load_text_model(self, model_path, num_epochs=5, lr=2e-5, patience=3):
        # Initialize BERT model
        bert_model = BertModel.from_pretrained("bert-base-uncased")

        # Check if the TextModel requires train_loader and val_loader for feature extraction
        try:
            text_model = self.text_model_class(
                num_epochs=num_epochs,
                lr=lr,
                patience=patience,
                model=bert_model,
                train_loader=None,  # Passing None for feature extraction
                val_loader=None,    # Passing None for feature extraction
                device=self.device
            )
        except TypeError:
            # Handle case where train_loader and val_loader are not required
            text_model = self.text_model_class(
                num_epochs=num_epochs,
                lr=lr,
                patience=patience,
                model=bert_model,
                device=self.device
            )

        text_model.model.load_state_dict(torch.load(model_path, map_location=self.device), strict=False)
        text_model.model.eval()
        return text_model


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from transformers import BertTokenizer, BertModel

class MultimodalTransformer(nn.Module):
    def __init__(self, audio_feature_dim, text_feature_dim, hidden_dim=512, num_heads=8, num_layers=4, output_dim=15111):
        super(MultimodalTransformer, self).__init__()
        # Audio embedding
        self.audio_fc = nn.Linear(audio_feature_dim, hidden_dim)
        # Text embedding
        self.text_fc = nn.Linear(text_feature_dim, hidden_dim)

        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=num_heads,
            dim_feedforward=hidden_dim * 2,
            dropout=0.1,
            activation='relu'
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # Output linear layer
        self.fc_out = nn.Linear(hidden_dim, output_dim)


    def forward(self, audio_features, text_features):
        # Project features to the same hidden dimension
        audio_embed = self.audio_fc(audio_features)
        text_embed = self.text_fc(text_features)

        # Combine along the feature dimension (dim=1)
        multimodal_input = torch.cat((audio_embed, text_embed), dim=1)  # (batch_size, seq_len, hidden_dim)

        # Pass through transformer encoder
        transformer_output = self.transformer(multimodal_input)

        # Use the first output token for the final prediction
        output = self.fc_out(transformer_output[:, 0, :])  # Use the [CLS] token from the sequence
        return output





In [ ]:

def load_models(audio_model_path, text_model_path, device = 'cuda'):
    # Load audio model
    audio_model = SpectrogramCNN()
    audio_model.load_state_dict(torch.load(audio_model_path, map_location=device))
    audio_model.eval().to(device)

    # Load text model
    text_model = TextModel(BertModel.from_pretrained('bert-base-uncased'), None, None, device)
    text_model.model.load_state_dict(torch.load(text_model_path, map_location=device))
    text_model.model.eval()

    return audio_model, text_model



In [ ]:
from transformers import BertTokenizer

def preprocess_inputs(audio_input, text_input, tokenizer, device):
    # Process audio (assuming audio_input is a spectrogram tensor)
    audio_features = audio_input.to(device)

    # Process text (tokenize using BERT tokenizer)
    tokenized_text = tokenizer(text_input, padding=True, truncation=True, return_tensors='pt')
    input_ids = tokenized_text['input_ids'].to(device)
    attention_mask = tokenized_text['attention_mask'].to(device)

    return audio_features, input_ids, attention_mask


In [ ]:
def train_multimodal_transformer(
    multimodal_model, audio_model, text_model, dataloader, tokenizer, device, num_epochs=5, lr=1e-4
):
    optimizer = optim.Adam(multimodal_model.parameters(), lr=lr)
    criterion = nn.BCEWithLogitsLoss()

    multimodal_model.to(device)

    for epoch in range(num_epochs):
        multimodal_model.train()
        epoch_loss = 0

        for batch in dataloader:
            # Unpack batch
            spectrograms, captions, aspect_labels = batch
            spectrograms = spectrograms.to(device)
            aspect_labels = aspect_labels.to(device)

            # Extract features
            with torch.no_grad():
                audio_features = audio_model.encoder(spectrograms)
                text_features = text_model.model(input_ids=captions['input_ids'], attention_mask=captions['attention_mask'])[0][:, 0, :]  # [CLS] token embedding

            # Forward pass through multimodal transformer
            outputs = multimodal_model(audio_features, text_features)

            # Compute loss
            loss = criterion(outputs, aspect_labels)
            epoch_loss += loss.item()

            # Backward pass and optimization
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        print(f"Epoch {epoch + 1}/{num_epochs}, Loss: {epoch_loss:.4f}")


In [ ]:
def generate_caption(multimodal_model, audio_model, tokenizer, audio_input, device):
    # Extract audio features
    with torch.no_grad():
        audio_features = audio_model.encoder(audio_input.to(device))

    # Create placeholder text features (e.g., zero embeddings)
    text_features = torch.zeros_like(audio_features).to(device)

    # Predict with multimodal model
    with torch.no_grad():
        output = multimodal_model(audio_features, text_features)

    # Decode output into a caption
    predicted_caption = tokenizer.decode(torch.argmax(output, dim=1).tolist(), skip_special_tokens=True)
    return predicted_caption



In [ ]:
from tqdm import tqdm

def train_and_save_model(
    multimodal_model,
    audio_model,
    text_model,
    train_loader,
    val_loader,
    tokenizer,
    device,
    num_epochs=5,
    lr=1e-4,
    save_path="multimodal_transformer.pth"
):
    optimizer = optim.Adam(multimodal_model.parameters(), lr=lr)
    criterion = nn.BCEWithLogitsLoss()

    multimodal_model.to(device)

    best_val_loss = float("inf")

    for epoch in range(num_epochs):
        print(f"\nEpoch {epoch + 1}/{num_epochs}")

        # Training phase
        multimodal_model.train()
        train_loss = 0

        with tqdm(train_loader, desc="Training", unit="batch") as pbar:
            for spectrograms, encoded_caption, aspect_labels in pbar:
                # Move data to device
                spectrograms = spectrograms.to(device)
                aspect_labels = aspect_labels.to(device)
                captions = {
                    "input_ids": encoded_caption["input_ids"].squeeze(1).to(device),
                    "attention_mask": encoded_caption["attention_mask"].squeeze(1).to(device),
                }

                # Extract features using pre-trained models
                with torch.no_grad():
                    audio_features = audio_model(spectrograms)
                    text_features = text_model.model(
                        input_ids=captions["input_ids"],
                        attention_mask=captions["attention_mask"],
                    )[0][:, 0, :]  # Extract [CLS] token embedding

                # Forward pass through the multimodal model
                outputs = multimodal_model(audio_features, text_features)

                # Compute loss
                loss = criterion(outputs, aspect_labels)
                train_loss += loss.item()

                # Backward pass and optimization
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

                # Update progress bar
                pbar.set_postfix(loss=loss.item())

        avg_train_loss = train_loss / len(train_loader)
        print(f"Train Loss: {avg_train_loss:.4f}")

        # Validation phase
        multimodal_model.eval()
        val_loss = 0

        with torch.no_grad():
            with tqdm(val_loader, desc="Validation", unit="batch") as pbar:
                for spectrograms, encoded_caption, aspect_labels in pbar:
                    # Move data to device
                    spectrograms = spectrograms.to(device)
                    aspect_labels = aspect_labels.to(device)
                    captions = {
                        "input_ids": encoded_caption["input_ids"].squeeze(1).to(device),
                        "attention_mask": encoded_caption["attention_mask"].squeeze(1).to(device),
                    }

                    # Extract features
                    audio_features = audio_model(spectrograms)
                    text_features = text_model.model(
                        input_ids=captions["input_ids"],
                        attention_mask=captions["attention_mask"],
                    )[0][:, 0, :]  # Extract [CLS] token embedding

                    # Forward pass
                    outputs = multimodal_model(audio_features, text_features)

                    # Compute loss
                    loss = criterion(outputs, aspect_labels)
                    val_loss += loss.item()

                    # Update progress bar
                    pbar.set_postfix(loss=loss.item())

        avg_val_loss = val_loss / len(val_loader)
        print(f"Val Loss: {avg_val_loss:.4f}")

        # Save model if validation loss improves
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(multimodal_model.state_dict(), save_path)
            print(f"Model saved to {save_path}")



# Load model for inference
def load_multimodal_model(model_path, multimodal_model, device="cuda"):
    multimodal_model.load_state_dict(torch.load(model_path, map_location=device))
    multimodal_model.to(device)
    multimodal_model.eval()
    print(f"Model loaded from {model_path}")
    return multimodal_model


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Load the CSV file
csv_file = 'resources/musiccaps-public.csv'
data = pd.read_csv(csv_file)


# Build a vocabulary of unique aspects from the training set
all_aspects = set()
for aspects in data['aspect_list']:
    aspect_list = eval(aspects)  # Convert string representation to list
    all_aspects.update(aspect_list)

aspect_to_index = {aspect: i for i, aspect in enumerate(sorted(all_aspects))}
num_aspects = len(aspect_to_index)  # Total number of unique aspects
print(f"Total unique aspects: {num_aspects}")

Training set size: 4416, Validation set size: 1105
Total unique aspects: 11349


In [ ]:
class MultimodalModelTrainer:
    def __init__(self, multimodal_model_class, audio_model, text_model, device="cuda"):
        self.multimodal_model_class = multimodal_model_class
        self.audio_model = audio_model
        self.text_model = text_model
        self.device = device

    def train_and_save(self, train_loader, val_loader, tokenizer, num_epochs=10, lr=1e-4, save_path="multimodal_transformer.pth"):
        multimodal_model = self.multimodal_model_class(audio_feature_dim=512, text_feature_dim=768, output_dim=num_aspects).to(self.device)
        # Train and save logic here
        train_and_save_model(
            multimodal_model,
            self.audio_model,
            self.text_model,
            train_loader,
            val_loader,
            tokenizer,
            device=self.device,
            num_epochs=num_epochs,
            lr=lr,
            save_path=save_path
        )

    def load_model(self, model_path):
        trained_model = load_multimodal_model(model_path, self.multimodal_model_class, device=self.device)
        return trained_model


In [ ]:
sample = train_dataset[0]
print(type(sample))
print(sample)

In [ ]:
for batch in train_loader:
    print(batch.keys())
    print(batch['input_ids'].shape)
    print(batch['labels'].shape)
    break

In [ ]:
# Configurations (paths, tokenizer, etc.)
csv_file = './musiccaps-public.csv'
spectrogram_train_dir = 'resources/training_spectrograms'
spectrogram_val_dir = 'resources/testing_spectrograms'
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
aspect_to_index = {aspect: i for i, aspect in enumerate(sorted(all_aspects))}

# DataHandler - Loading and splitting data
data_handler = DataHandler(csv_file, spectrogram_val_dir, tokenizer, aspect_to_index)

# DataHandler takes the following
# csv_file, spectrogram_val_dir, tokenizer, aspect_to_index, test_size
# so why are we passing both the spectrogram train dir and val dir?
# the reason that line 9 was erroring was because the arguments were off by one, so the aspect_to_index was being passed where the test_size should have been
# and, in this case, we don't need to specify the test size, because it is already set to 0.2 by the function

train_dataset, val_dataset = data_handler.create_datasets()
train_loader, val_loader = data_handler.create_dataloaders(train_dataset, val_dataset)

# ModelHandler - Loading models
model_handler = ModelHandler(SpectrogramCNN, TextModel, device="cuda")
audio_model = model_handler.load_audio_model("best_audio_model.pth")
text_model = model_handler.load_text_model("best_model.pth")

# MultimodalModelTrainer - Training and saving the multimodal model
trainer = MultimodalModelTrainer(MultimodalTransformer, audio_model, text_model, device="cuda")
trainer.train_and_save(train_loader, val_loader, tokenizer, num_epochs=10, lr=1e-4, save_path="multimodal_transformer.pth")

# Load and use the trained model
trained_model = trainer.load_model("multimodal_transformer.pth")

### FULL PIPELINE!
This is a work in progress proof of concept type pipeline where it takes all 5 (yes 5) models and uses them to make a prediction on the caption for the given audio input.

In [ ]:
# Load Model 1
model_1 = SpectrogramCNN()
model_1.load_state_dict(torch.load('model_1.pth'))
model_1.eval()

file_path = "pth/to/your/audio/file"

wav_file = convert_audio(file_path)
non_norm_spectrogram = convert_to_stft_spectrograms(wav_file)
spectrogram = normalize_spectrograms(non_norm_spectrogram)

# Extract latent features
with torch.no_grad():
    processed_spectrogram = model_1.encoder(spectrogram.unsqueeze(0))  # Shape: (batch_size, compressed_dim)

In [2]:
# Load Model 2
model_2 = SpectrogramAspectClassifier()
model_2.load_state_dict(torch.load('model_2.pth'))
model_2.eval()

# Predict aspects
with torch.no_grad():
    aspect_predictions = model_2(processed_spectrogram)  # Shape: (batch_size, num_aspects)
    binary_aspects = (aspect_predictions > 0.5).float()  # Thresholded binary predictions

NameError: name 'AspectClassifierModel' is not defined

In [ ]:
# Load Model 3
model_3 = CaptionGenerator(num_aspects=11349, vocab_size=tokenizer.vocab_size, max_length=128)
model_3.load_state_dict(torch.load('model_3.pth'))
model_3.eval()

# Dummy inputs for BERT placeholders
dummy_input_ids = torch.zeros((1, 128), dtype=torch.long)
dummy_attention_mask = torch.ones_like(dummy_input_ids)

# Generate caption
with torch.no_grad():
    outputs = model_3(binary_aspects, dummy_input_ids, dummy_attention_mask)
    predicted_token_ids = outputs.argmax(dim=-1)
    caption = tokenizer.decode(predicted_token_ids[0], skip_special_tokens=True)

print("Generated Caption:", caption)

In [ ]:
# Load Model 4
model_4 = TemporalAlignmentModel()
model_4.load_state_dict(torch.load('model_4.pth'))
model_4.eval()

# Predict temporal alignment
with torch.no_grad():
    alignment_map = model_4(processed_spectrogram, binary_aspects)  # Shape: (batch_size, time_steps, num_aspects)

In [ ]:
# Load Model 5
model_5 = FinalIntegrationModel()
model_5.load_state_dict(torch.load('model_5.pth'))
model_5.eval()

# Final integration
with torch.no_grad():
    final_output = model_5(caption, alignment_map)
    print(final_output)  # Final description with temporal annotations